# Detect object with YOLOv5 and OpenCV

## Sources:
- [Object Detection using YOLOv5 and OpenCV DNN in C++ and Python](https://learnopencv.com/object-detection-using-yolov5-and-opencv-dnn-in-c-and-python/?ck_subscriber_id=1558025914)

## Import modules

In [48]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from datetime import date


# import 3rd-party modules
import cv2
import numpy as np


# import local modules
from utils.renderer.giffer import create_gif
from utils.renderer.videographer import create_video
from utils.project_manager import Project
from utils.renderer.resizer import get_interpolation
from utils.renderer.resizer import resize_with_pad, resize_with_crop

## Define Global Parameters

In [2]:
# set blob size (blob: binary large object; contains the data in readable raw format;
# image has to be converted to a blob so as the network can process it)
INPUT_HEIGHT, INPUT_WIDTH = 640, 640

# set low probability class filter
SCORE_THRESHOLD = 0.5

# set overlapping bounding boxes filter
NMS_THRESHOLD = 0.45

# set low probability detection filter
CONFIDENCE_THRESHOLD = 0.45

# set text parameters
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.7
THICKNESS = 1

# set colors
BLACK = (0,0,0)
BLUE = (255, 178, 50)
YELLOW = (0,255, 255)

## Define functions

In [42]:
def draw_label(img, label, x, y):
    """
    Function to draw text onto image at xy coords
    """
    # get text size
    text_size = cv2.getTextSize(label, FONT, FONT_SCALE, THICKNESS)
    dim, baseline = text_size[0], text_size[1]

    # use text size to create a black rectangle
    cv2.rectangle(img, (x,y), (x + dim[0], y + dim[1] + baseline), BLACK, cv2.FILLED)

    # display text inside rectangle
    cv2.putText(img, label, (x, y + dim[1]), FONT, FONT_SCALE, YELLOW, THICKNESS, cv2.LINE_AA)

def pre_process(input_img, net):
    """
    Function to 
    - convert image to a blob of a 4D array object
    - pass it to the neural network

    Returns a 2D array of shape (25200, 85) or from OpenCV-Python 4.5.5, a 3D array of shape (1, 25200, 85)
    - rows = number of detections (=> number of bounding boxes; 25200 detections)
    - columns = 85 info of each detection: x, y, width, height, confidence, class scores of 80 classes
        * x,y : normalized center coords of detected bounding box
        * width, height: normalized width and height
        * confidence: probability of detection being an object
        * class scores of 80 objects from COCO dataset 2017 (on which model has been trained)
    """
    # create a 4d blob from a frame
    blob = cv2.dnn.blobFromImage(input_img, 1/255, (INPUT_WIDTH, INPUT_HEIGHT), [0,0,0], 1, crop=False)

    # set input to the network
    net.setInput(blob)

    # run the forward pass to get output of the output layers
    outputs = net.forward(net.getUnconnectedOutLayersNames())

    return outputs

def post_process(input_img, outputs, label_names):
    """
    Function to unwrap the outputs from the neural network
    """
    # create empty lists to store the different outputs from the network
    class_ids = []
    confidences = []
    boxes = []

    # get number of detections (i.e. 25200)
    nb_detections = outputs[0].shape[1]

    # get image shape
    img_height, img_width = input_img.shape[:2]

    # get resizing weights
    x_factor = img_width / INPUT_WIDTH
    y_factor =  img_height / INPUT_HEIGHT

    # iterate through 25200 detections
    for detection_idx in range(nb_detections):
        
        # get detection
        detection = outputs[0][0][detection_idx]
        confidence = detection[4]
        
        # discard bad detections and continue
        if confidence >= CONFIDENCE_THRESHOLD:

                # get the scores of the 80 classes
                classes_scores = detection[5:]
                
                # get index of best class score
                class_id = np.argmax(classes_scores)

                #  continue if the class score is above threshold
                if (classes_scores[class_id] > SCORE_THRESHOLD):

                    # append confidence and class ids to their respective list
                    confidences.append(float(confidence)) # convert confidence to float for NMSBoxes fct
                    class_ids.append(class_id)

                    # unwrap detection outputs
                    cx, cy, w, h = detection[0], detection[1], detection[2], detection[3]

                    # build bbox, resized according to input img
                    left = int((cx - w/2) * x_factor)
                    top = int((cy - h/2) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)
                    box = np.array([left, top, int(left + width), int(top + height)])
                    boxes.append(box)

    # perform non maximum suppression to eliminate redundant, overlapping boxes with lower confidences
    indices = cv2.dnn.NMSBoxes(boxes, confidences, CONFIDENCE_THRESHOLD, NMS_THRESHOLD)

    # iterate through index of remaining bboxes
    for i in indices.reshape(-1):

        # get bbox and its properties
        box = boxes[i]
        left = box[0]
        top = box[1]
        width = box[2]
        height = box[3]

        # draw bbox       
        cv2.rectangle(input_img, (left, top), (left + width, top + height), BLUE, 3*THICKNESS)

        # get class label                   
        label = f"{label_names[class_ids[i]]}:{confidences[i]:.2f}"

        # draw label             
        draw_label(input_img, label, left, top)
        
    return input_img

## Define main function

In [4]:
def main(img_path, model_path, label_names_path):
    """
    Function to load model, label names, read image, 
    perform pre-processing and post-processing 
    followed by displaying efficiency information.
    """

    # load model with weights
    net = cv2.dnn.readNet(model_path)

    # get label names
    with open(label_names_path, 'rt') as f:
        label_names = f.read().rstrip('\n').split('\n')

    # read img
    frame = cv2.imread(img_path)

    # process img
    detections = pre_process(frame, net)
    img = post_process(frame.copy(), detections, label_names)

    # add efficiency information: 
    # the fct getPerfProfile returns overall time for inference (t)
    # and the timings for each of the layers (in layersTimes).
    t, _ = net.getPerfProfile()
    label = f'Inference time:{(t * 1000.0 /  cv2.getTickFrequency()):.2f} ms'
    print(label)
    cv2.putText(img, label, (20, 40), FONT, FONT_SCALE,  (0, 0, 255), THICKNESS, cv2.LINE_AA)

    # show img
    cv2.imshow('Output', img)

    # wait for any press on keyboard
    cv2.waitKey(0)

    # destroy all windows
    cv2.destroyAllWindows()
    cv2.waitKey(1) # workaround on mac to effectively close the windows

In [56]:
def main_video(video_path, model_path, label_names_path, out_path, out_path_shape=None, resize_fct=None, rotate_90_multiple=None):
    """
    Function to load model, label names, read video, 
    perform pre-processing and post-processing 
    add efficiency information,
    save output video

    """
    
    # load model with weights
    net = cv2.dnn.readNet(model_path)

    # get label names
    with open(label_names_path, 'rt') as f:
        label_names = f.read().rstrip('\n').split('\n')


    # Initialize video stream
    video_cap = cv2.VideoCapture(str(video_path))

    # get video parameters
    video_nb_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = video_cap.get(cv2.CAP_PROP_FPS)
    video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"number of frames = {video_nb_frames}")
    print(f"fps = {video_fps}")
    print(f"video width = {video_width}")
    print(f"video height = {video_height}")

    # set codec for output video
    codec = "H264"

    if out_path_shape is not None:
        # set output shape
        out_height, out_width, out_channel = out_path_shape
    else:
        out_height, out_width, out_channel = video_height, video_width, 3

    # create a videoWriter object
    fourcc = cv2.VideoWriter_fourcc(*codec)
    out_video = cv2.VideoWriter(filename=str(out_path), fourcc=fourcc, fps=video_fps, frameSize=(out_width, out_height))

    # iterate over all frames
    while True:

        # read video stream
        ret, frame = video_cap.read()

        if not ret:
            print("frame is empty")
            break

        # process img
        detections = pre_process(frame, net)
        img = post_process(frame.copy(), detections, label_names)

        # add efficiency information: 
        # the fct getPerfProfile returns overall time for inference (t)
        # and the timings for each of the layers (in layersTimes).
        t, _ = net.getPerfProfile()
        label = f'Inference time:{(t * 1000.0 /  cv2.getTickFrequency()):.2f} ms'
        print(label)
        cv2.putText(img, label, (20, 40), FONT, FONT_SCALE,  (0, 0, 255), THICKNESS, cv2.LINE_AA)

        # rotate & resize frame if asked
        if rotate_90_multiple is not None:
            img = np.rot90(img, rotate_90_multiple)
        
        if resize_fct is not None:
            img = resize_fct(img, ref_img_shape=(out_height, out_width, out_channel))

        # write output frame
        out_video.write(img)

    # release video stream & video rendering
    video_cap.release()
    out_video.release()

## Run main function

In [44]:
if __name__ == '__main__':
    img_path = "/Users/derrickvanfrausum/Pictures/carnival.png"
    model_path = "assets/models/yolov5x6.onnx"
    label_names_path = "assets/models/coco.names"

    main(img_path, model_path, label_names_path)

Inference time:1955.00 ms


In [57]:
if __name__ == '__main__':

    # create project
    project = Project(project_dir="assets/images/yolov5")

    # get current date
    today = date.today().strftime("%Y%m%d")

    # set input & output video path
    video_path = Path("assets/images/yolov5/carnival_trim.mp4")
    out_path = project.project_dir / f"{video_path.stem}_{today}.mp4"

    model_path = "assets/models/yolov5x6.onnx"
    label_names_path = "assets/models/coco.names"

    out_path_shape = (1920, 1080, 3)

    main_video(video_path, model_path, label_names_path, out_path, out_path_shape, resize_with_pad, rotate_90_multiple=-1)

number of frames = 93
fps = 24.21875
video width = 3840
video height = 2160
Inference time:2443.50 ms
Inference time:2259.18 ms
Inference time:2548.73 ms
Inference time:1417.79 ms
Inference time:1198.18 ms
Inference time:1902.49 ms
Inference time:2103.41 ms
Inference time:1202.64 ms
Inference time:1226.91 ms
Inference time:1147.50 ms
Inference time:1175.59 ms
Inference time:1163.21 ms
Inference time:1242.42 ms
Inference time:1912.52 ms
Inference time:1339.85 ms
Inference time:1997.17 ms
Inference time:2543.66 ms
Inference time:1455.58 ms
Inference time:1304.55 ms
Inference time:1404.75 ms
Inference time:1267.25 ms
Inference time:2321.82 ms
Inference time:1228.61 ms
Inference time:1277.07 ms
Inference time:2227.94 ms
Inference time:2532.60 ms
Inference time:2822.53 ms
Inference time:2126.65 ms
Inference time:3405.38 ms
Inference time:1660.03 ms
Inference time:1633.91 ms
Inference time:2496.77 ms
Inference time:1431.47 ms
Inference time:1209.45 ms
Inference time:1235.05 ms
Inference time